# 추론 모델 밑바닥부터 6편 실습 노트북 — 7장: 강화학습 들여다보기(기록·정책 비율 자르기·KL 항) (한국어 주석판)

**출처:** 원본 노트북 `ch07_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch07/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** 학습은 GPU가 있어야 현실적이고 시간도 오래 걸립니다. 처음에는 스텝 수(`--steps`)를 작게 잡아 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 7장: 강화학습을 위한 GRPO 개선하기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F01_raschka.webp?1" width=600>

&nbsp;
## 7.1 GRPO 개선하기

- 참고: 이 장이 너무 기술적이고 GRPO의 세부 내용에 관심이 없다면 건너뛰어도 됩니다. 다음 장은 이 장의 내용에 기대지 않습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F02_raschka.webp?1" width=400>

&nbsp;
## 7.2 GRPO 성능 지표 기록하기

- 앞 장에서 짧게 RLVR 학습을 돌려 보고 결과를 간단히 살펴봤습니다.

- 예를 들어 앞 절에서 짧은 학습의 출력을 봤는데, 생김새는 아래와 같았습니다.

```
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=5.5
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.8
[Step 3/50] loss=0.3592 reward_avg=0.250 avg_resp_len=7.8
[Step 4/50] loss=2.7401 reward_avg=0.250 avg_resp_len=56.5
[Step 5/50] loss=3.3214 reward_avg=0.500 avg_resp_len=251.2
# ...
```

- 이 절에서는 거기서 이어 갑니다. RLVR을 돌릴 때 지켜볼 만한 지표 몇 가지를 알아봅니다.

&nbsp;
### 7.2.1 GRPO 학습 돌리기

- 6장의 코드는 GRPO 전체 흐름을 보여 주면서도 읽기 쉽게 짧게 쓰려고, 둘 사이에서 균형을 잡았습니다.
- 부록 자료에는 같은 코드를 담은 스크립트([ch06/02_rlvr_grpo_scripts_intro
/rlvr_grpo_original_no_kl.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py))가 있습니다. 터미널에서 돌리기에는 이쪽이 더 편할 수 있습니다.
- 이 장에서는 편의를 위해 부록 자료의 스크립트를 가져다 씁니다. 노트북과 본문에 같은 코드를 잔뜩 되풀이하면 어수선해지기 때문입니다.
- 아래 도우미 함수로 이 장 내내 부록 자료의 코드를 내려받아 지금 폴더에 저장합니다.

In [ ]:
from pathlib import Path
import requests

def download_from_github(rel_path, out=None):
    # 부록 파일을 가져올 깃허브 원본 주소의 앞부분입니다
    github_raw_base = (  # Base URL
        "https://raw.githubusercontent.com/rasbt/"
        "reasoning-from-scratch/refs/heads/main/"
    )

    rel_path = Path(rel_path)
    # Use URL file name as default output file name
    out = Path(out) if out is not None else Path(rel_path.name)

    # Skip download if file already exists locally
    if out.exists():
        size_kb = out.stat().st_size / 1e3
        print(f"{out}: {size_kb:.1f} KB (cached)")
        return out

    # Download file
    r = requests.get(github_raw_base + rel_path.as_posix())
    r.raise_for_status()

    # 받은 내용을 파일로 저장하고 크기를 보여 줍니다
    out.write_bytes(r.content)
    size_kb = out.stat().st_size / 1e3
    print(f"{out}: {size_kb:.1f} KB")

- GRPO 스크립트(6장 코드 기반)를 내려받습니다.

In [ ]:
# 6장의 GRPO 학습 스크립트를 내려받습니다
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

- 내려받은 파일 크기가 위에 나온 값보다 훨씬 작다면, 내려받기에 문제가 있었을 수 있습니다.
- URL을 바르게 입력했는지 다시 확인해 보세요.
- 그래도 내려받기가 계속 안 되면 [문제 해결 안내(Troubleshooting Guide)](https://github.com/rasbt/reasoning-from-scratch/blob/main/troubleshooting.md)를 참고하세요.

- 학습 코드는 비용이 꽤 들어서 직접 돌리지 않으려는 것도 충분히 이해됩니다. 그래서 아래 실행의 로그 파일을 부록 자료로 드립니다(내려받는 법은 조금 뒤에 나옵니다).


```bash
uv run rlvr_grpo_original_no_kl.py \
    --steps 500  \
    --max_new_tokens 1024
```

- **팁:** 위 실행 명령에 `--show_eta`를 붙이면, 여러분 컴퓨터에서 스크립트 전체가 얼마나 걸릴지 예상 시간을 보여 줍니다.

- 스크립트는 위 명령처럼 터미널에서 돌리기를 권합니다. 주피터 노트북의 코드 셀에서 앞에 "!"를 붙여 돌릴 수도 있습니다. 예: `!uv run rlvr_grpo_original_no_kl.py`
- `uv`를 쓰지 않는다면 `uv run` 대신 `python`을 쓰세요. 예: `python rlvr_grpo_original_no_kl.py`

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F03_raschka.webp" width=600>

- 앞에서 말했듯이, 비용이 큰 위 실행을 직접 재현하는 대신 아래처럼 로그 파일을 내려받을 수 있습니다.

In [ ]:
# Log file in plain text

download_from_github(
    "ch07/02_logs/ch06_rlvr_grpo_original_no_kl_metrics.txt"
)

# Same log file in CSV format
download_from_github(
    "ch07/02_logs/ch06_rlvr_grpo_original_no_kl_metrics.csv"
)

&nbsp;
### 7.2.2 GRPO 학습 결과 살펴보기

- 이 장 내내 아래 도우미 함수로 로그 파일의 결과를 그래프로 그립니다.

In [ ]:
import csv
import matplotlib.pyplot as plt


def moving_average(values, window_fraction=0.25):
    # Smooth a noisy training signal to reveal longer-term trends during training
    # 전체 길이의 25%를 이동 평균 창 크기로 씁니다
    window_size = max(1, int(window_fraction * len(values)))
    smoothed = []

    for i in range(len(values)):
        start_idx = max(0, i - window_size + 1)
        window_mean = sum(values[start_idx : i + 1]) / (i - start_idx + 1)
        smoothed.append(window_mean)

    return smoothed


def plot_grpo_metrics(csv_path, columns, save_as=None):
    # 지표마다 스텝 목록과 값 목록을 따로 모읍니다
    data = {name: {"steps": [], "values": []} for name in columns}

    # Open and read CSV log file
    with Path(csv_path).open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row or not row.get("step"):
                continue

            # Use the training step as the shared x-axis across all metrics
            step = int(row["step"])

            for name in columns:
                value_str = row.get(name)
                if value_str:
                    data[name]["steps"].append(step)
                    data[name]["values"].append(float(value_str))

    # Create a fixed grid so loss, rewards, response length, etc. can be shown side by side
    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(6, 4))
    axes = axes.ravel()

    for i, name in enumerate(columns):
        steps = data[name]["steps"]
        values = data[name]["values"]

        # Skip metrics that are not present
        if not values:
            fig.delaxes(axes[i])
            continue

        # Evaluation accuracy as barplot because we don't have data for each step
        if name == "eval_acc":
            axes[i].bar(steps, values, width=20)
        else:
            # 흐린 선은 원래 값, 진한 선은 이동 평균입니다
            axes[i].plot(steps, values, alpha=0.4)
            axes[i].plot(steps, moving_average(values))

        axes[i].set_ylabel(name)

    for j in (2, 3):
        if axes[j] in fig.axes:
            axes[j].set_xlabel("Step")

    plt.tight_layout()
    if save_as is not None:
        plt.savefig(save_as)
    plt.show()


# Plot the GRPO training run
# 손실·평균 보상·응답 길이·평가 정확도를 한 번에 그립니다
plot_grpo_metrics(
    "ch06_rlvr_grpo_original_no_kl_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    # save_as="4.pdf"
)

- 전체적으로 눈에 띄는 점과 생각을 정리해 봅니다.
  - 평균 응답 길이는 처음에 늘어나야 합니다(정확도도 함께 오르면 가장 좋습니다). 전반적으로 괜찮아 보입니다. 다만 400 스텝 직전에 조금 줄어드는 모습이 보입니다.
  - 사전학습과 달리 여기서는 손실(loss) 값이 크게 의미 있지 않습니다. 대체로 평평하게 머물러야 하고, 어느 정도 출렁이는 것은 괜찮습니다(그래도 끝 무렵에 튀는 값은 조금 걱정스럽습니다).
- 평균 보상은 올라가야 합니다. 엄밀히 말해 평균 보상 1.00은 모든 답이 맞았다는 뜻이라 아주 좋습니다. 하지만 그 지점에서는 더 배울 학습 신호가 없으니 학습을 멈춰야 합니다.
- 목표 과제(외부 과제) 성능, 여기서는 MATH-500 정확도가 올라야 합니다. 처음에는 오르다가 나중에는 떨어집니다.

- 요약하면, 초반에 빠르게 좋아지다가 갈수록 효과가 줄어듭니다.
  - 한 가지 이유는 알고리즘이 그다지 안정적이지 않다는 점입니다.
  - (문제가 점점 어려워져서라고 볼 수도 있습니다. 하지만 그렇다면 평가 정확도가 떨어지지는 않았을 것입니다.)

- 여기서는 GRPO 학습을 단순하게 기록하는 데 집중합니다. 다음 절에서 지표를 몇 가지 더합니다.
- 참고로 평가 정확도(여기서는 MATH-500 정확도)는 기본으로 기록되지 않습니다.
- 학습 중에 `--eval_on_checkpoint 500`을 붙이면 MATH-500 테스트 문제 500개로 계산할 수 있습니다. 다만 학습이 느려집니다.
- 아니면 학습이 끝난 뒤 `evaluate_math500.py` 스크립트(3장 코드가 들어 있습니다)로 계산할 수도 있습니다.

In [ ]:
# 3장의 MATH-500 평가 스크립트를 내려받습니다
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)

- 그러면 원하는 체크포인트로 아래처럼 평가를 돌릴 수 있습니다.

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --checkpoint_path \
    "checkpoints/rlvr_grpo_original_no_kl/\
qwen3-0.6B-rlvr-grpo-step00050.pth"
```

- 50 스텝마다 돌린 이 평가 결과는 앞에서 살펴본 로그 파일에 이미 넣어 두었습니다.

&nbsp;
## 7.3 더 깊은 GRPO 성능 지표 기록하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F05_raschka.webp" width=400>

- 앞 절에서는 GRPO 학습을 지켜보는 기본 성능 지표를 살펴봤습니다.
- 이 밖에도 기록해 두면 쓸모 있는 지표가 여럿 있습니다.
- 두 가지 예를 들면 다음과 같습니다.
  - 6장의 `compute_grpo_loss` 함수에서 이미 계산한 어드밴티지(advantage)
  - 생성한 시퀀스의 엔트로피(entropy)

&nbsp;
### 7.3.1 어드밴티지 기록하기

- GRPO 알고리즘에서는 앞서(6장) 설명한 대로 이른바 어드밴티지를 계산합니다.
- 이 어드밴티지는 학습이 어떻게 흘러가는지 분석할 때도 쓸모가 있습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F06_raschka.webp" width=400>

- 특히 어드밴티지에서 요약 통계 두 가지를 새로 뽑습니다. 평균값(표본 평균 `mean`)과 표준편차(`std`)입니다.

In [ ]:
import torch

def compute_advantage_stats(rewards_list):
    # This is what we already compute in GRPO:
    rewards = torch.tensor(rewards_list)
    # 보상에서 평균을 빼고 표준편차로 나눕니다 (1e-4는 0으로 나누기 방지)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # These are the new statistics we add:
    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

- 아래는 보상 값 4개로 이뤄진 작은 예시입니다(앞 그림의 값과 비슷합니다).

In [ ]:
# 맞음 2개, 틀림 2개인 그룹으로 시험합니다
adv, adv_avg, adv_std = compute_advantage_stats([1., 1., 0., 0.])

print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

- 평균(Mean):
  - 우리가 어드밴티지를 계산하는 방식에서는 평균이 늘 0입니다. GRPO에서 평균은 코드가 뜻대로 돌아가는지 확인하는 점검용일 뿐입니다.

- 표준편차(Std):
  - 여기서는 표준편차가 더 많은 정보를 줍니다.
  - std ≈ 1이면 기울기(gradient) 신호의 크기가 알맞아서 업데이트가 안정적입니다.
  - std ≪ 1이면 신호가 사라지고 있다는 뜻입니다(보상이 무너질 때 자주 생깁니다).
  - std ≫ 1이면 업데이트가 너무 들쭉날쭉해서 학습이 흔들릴 수 있습니다.

- 극단적인 경우는 보상이 모두 비슷할 때입니다. 이때 표준편차는 0이 되므로, 학습 신호가 없다는 것을 알아채는 기준으로 쓸 수 있습니다.

In [ ]:
# 전부 틀린 경우: 표준편차가 0이 되어 학습 신호가 없습니다
adv, adv_avg, adv_std = compute_advantage_stats([0., 0., 0., 0.])
print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

In [ ]:
# 전부 맞힌 경우도 표준편차가 0입니다
adv, adv_avg, adv_std = compute_advantage_stats([1., 1., 1., 1.])
print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

- 실제로는 어드밴티지 통계를 평균 보상과 함께 보는 것이 가장 좋습니다. 앞에서 말했듯이 평균 보상 1은 사실 좋은 일이기 때문입니다. 학습 신호는 없어지지만, 모델이 모든 답을 맞히고 있다는 뜻이니까요.
- 이때는 학습을 멈추거나 더 어려운 문제를 골라야 합니다.

&nbsp;
### 7.3.2 엔트로피 기록하기

- 엔트로피는 LLM이 다음 토큰을 만들 때 얼마나 확신이 없는지를 잽니다.
  - 엔트로피가 높으면: 모델이 확률을 여러 토큰에 나눠 줍니다(탐색에 좋습니다).
  - 엔트로피가 낮으면: 모델이 거의 모든 확률을 한 토큰에 몰아줍니다(더 결정적으로 움직이고, 붕괴(collapse)의 신호일 수 있습니다).

- 엔트로피를 계산하기 전에, 5장에서 로그 확률 값(logprobs)을 어떻게 계산했는지 다시 떠올려 봅시다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F07_raschka.webp" width=600>

- 기본 모델에 프롬프트를 넣어 얻은 실제 로짓(logits) 대신, 어휘 크기가 7이라고 가정한 예시 로짓을 씁니다(원래 어휘 15만 1천 개를 그대로 쓰면 위 그림이 너무 어지러워집니다).

In [ ]:
# Toy logits
# 어휘가 7개라고 가정한 예시 로짓입니다
logits = torch.tensor([
    0.6667, -2.0000,  1.3333, -0.0000, -0.6667,  2.0000, -1.3333
])

- 아래 코드는 앞 그림의 계산을 그대로 구현합니다.

In [ ]:
# 로짓을 로그 확률로 바꿉니다
logprobs = torch.log_softmax(logits, dim=-1)
print("All token logprobs:", logprobs)

# Index of the selected token
selected_token = torch.argmax(logprobs)

# 가장 확률이 높은 토큰의 로그 확률을 꺼냅니다
selected = logprobs[selected_token]
print("Selected token ID:", selected_token)
print("Selected token logprob:", selected)

- 엔트로피는 이와 관련된 개념입니다. 다음 그림처럼 확률 값에 로그 확률 값을 곱한 뒤, 그 곱들을 모두 더해 계산합니다.
- 원칙적으로는 확률이나 로그 확률의 합처럼 더 단순한 값을 기록할 수도 있습니다. 하지만 엔트로피는 확률 분포의 불확실성을 재는 잘 정립되고 해석하기 쉬운 척도입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F08_raschka.webp" width=400>

- 참고로 대략적인 기준은 다음과 같습니다.
  - 아주 낮은 엔트로피(≈ 0–0.5): 토큰 하나가 분포를 지배합니다. 모델이 매우 확신하고 거의 결정적으로 움직입니다.
  - 중간 엔트로피(≈ 1–2): 확률이 몇 개 토큰에 나뉩니다. 안정적으로 학습할 때 흔히 보이는 모습입니다.
  - 높은 엔트로피(≫ 2, log(어휘 크기)에 가까워짐. 여기서는 log(7) = 1.9459): 확률이 많은 토큰에 퍼집니다. 모델이 매우 불확실하고 거의 무작위처럼 움직입니다.

- 엔트로피는 아래처럼 계산할 수 있습니다.

In [ ]:
# Probabilities from logits
probs = torch.softmax(logits, dim=-1)
logprobs = torch.log_softmax(logits, dim=-1)

# Entropy from probabilities
# 엔트로피 = -Σ p·log p
entropy = torch.sum(-(probs * logprobs))

print("Probs:", probs)
print("Entropy:", entropy)

- 로그 확률 값만 있다면 `torch.exp`로 로그 확률을 다시 확률로 바꿀 수도 있습니다.
- 아래 값이 위의 "probs" 값과 같은 것을 볼 수 있습니다.

In [ ]:
# 로그 확률에 exp를 씌우면 다시 확률이 됩니다
print("Probs:", torch.exp(logprobs))

- 위 개념을 써서 앞 장의 `sequence_logprob` 함수를 `sequence_logprob_and_entropy` 함수로 업그레이드합니다. 이 함수는 로그 확률과 엔트로피를 함께 돌려줍니다.

In [ ]:
def sequence_logprob_and_entropy(model, token_ids, prompt_len):
    # Old: Code is identical to chapter 5
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # 각 위치에서 실제로 나온 다음 토큰의 로그 확률만 고릅니다
    targets = token_ids[1:]
    selected = logprobs[:-1].gather(1, targets.unsqueeze(-1)).squeeze(-1)

    # Log-prob of the generated answer tokens (sum over answer steps)
    selected_answer_logprobs = selected[prompt_len - 1:]
    logp_all_steps = torch.sum(selected_answer_logprobs)

    # New: Calculate entropy
    # 답 부분의 어휘 전체 분포를 가져옵니다
    all_answer_logprobs = logprobs[:-1][prompt_len - 1:]
    if all_answer_logprobs.numel() == 0:  # Safeguard if the model immediately returns EOS token
        entropy_all_steps = logp_all_steps.new_tensor(0.0)
    else:
        all_answer_probs = torch.exp(all_answer_logprobs)  # convert logprob to prob
        plogp = all_answer_probs * all_answer_logprobs     # elementwise p * log p
        step_entropy = -torch.sum(plogp, dim=-1)           # sum over vocab -> entropy per step
        entropy_all_steps = torch.mean(step_entropy)       # average over answer steps

    return logp_all_steps, entropy_all_steps

- 이 함수를 `compute_grpo_loss` 함수 안에서 쓰면 엔트로피도 함께 계산해 학습 중에 기록할 수 있습니다.
- 앞 그림은 LLM 롤아웃(rollout)에서 토큰 하나의 엔트로피(`step_entropy`)를 계산하는 모습이었습니다. `sequence_logprob_and_entropy`는 답의 모든 토큰에 대한 평균 엔트로피를 계산합니다: `entropy_all_steps = torch.mean(step_entropy)`
- 예를 들어 답이 `"this is the LLM response"`라면, 스텝 엔트로피는 토큰 하나(`"this"` 같은)의 엔트로피입니다. 답 전체 스텝의 평균 엔트로피는 답의 모든 토큰 `"this is the LLM response"`에 대한 엔트로피의 평균입니다.

&nbsp;
### 7.3.3 GRPO 지표 더 그려 보기

- 실제로 돌려 보고, 학습 한 번의 어드밴티지 통계와 엔트로피를 분석해 봅시다.
- 그러려면 `compute_grpo_loss` 안의 함수를 `sequence_logprob_and_entropy`로 바꾸고, `compute_grpo_loss`를 안에서 쓰는 `train_rlvr_grpo`에서 엔트로피를 출력하고 기록하면 됩니다.
- 긴 코드를 되풀이하지 않도록, 고친 전체 코드는 부록 자료에 있습니다. 아래처럼 내려받을 수 있습니다.

In [ ]:
# 어드밴티지·엔트로피 기록을 더한 스크립트를 내려받습니다
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_3_plus_tracking.py"
)

- 앞에서 다룬 학습 스크립트와 비슷하게 돌릴 수 있습니다.

```bash
uv run 7_3_plus_tracking.py \
    --steps 500  \
    --max_new_tokens 1024
```

- 학습이 오래 걸리므로, 결과로 나온 CSV 로그 파일을 아래처럼 내려받을 수 있습니다.

In [ ]:
# 미리 돌려 둔 학습 로그(CSV)를 내려받습니다
download_from_github(
    "ch07/02_logs/7_3_plus_tracking_metrics.csv"
)

In [ ]:
# 평균 보상과 어드밴티지 통계, 엔트로피를 함께 그립니다
plot_grpo_metrics(
    "7_3_plus_tracking_metrics.csv",
    columns=["reward_avg", "adv_avg", "adv_std", "entropy_avg"],
    # save_as="9.pdf"
)

- 평균 어드밴티지는 0이고 계속 0에 머뭅니다. GRPO 방식의 정규화에서 기대하는 그대로입니다.
  - 어드밴티지는 그룹 평균을 기준으로 계산하므로, 구조상 합이 0이 되어야 합니다.
  - 이 지표는 GRPO에서 주로 점검용입니다. 0에서 벗어난다면 버그나 다른 정규화 편향 문제가 있다는 뜻입니다.
- 어드밴티지 표준편차가 더 많은 것을 알려 줍니다.
  - 초반에는 분산이 비교적 큽니다. 모델이 다양한 롤아웃을 만든다는 뜻입니다.
  - 나중에는 어드밴티지 표준편차가 천천히 줄다가 자리를 잡습니다. 롤아웃들의 품질이 서로 비슷해진다는 뜻입니다.
  - 핵심은, 어드밴티지 표준편차가 0이 아니고 안정적이면 아직 쓸 만한 학습 신호가 남아 있다는 것입니다.

- 다음으로 엔트로피를 봅시다.
  - 학습 초반에는 엔트로피가 비교적 평평하고 낮습니다(LLM이 꽤 결정적으로 움직인다는 뜻입니다).
  - 실제로는 샘플링 온도(temperature)를 높이면 롤아웃 때 출력이 더 다양해집니다. 다만 LLM 자체의 엔트로피가 바뀌지는 않습니다.
  - 학습 후반(약 200 스텝 이후)에는 엔트로피가 크게 늘어납니다. 다음 토큰 분포가 넓어지고 모델이 더 무작위적으로 바뀐다는 뜻입니다.
  - 엔트로피가 아주 낮으면 붕괴의 신호일 수 있습니다. 모델이 같거나 아주 비슷한 출력을 되풀이하는 상태입니다.
  - 하지만 이번 학습에서는 엔트로피를 평균 보상, 사라지지 않은 어드밴티지 표준편차와 함께 보면 붕괴보다는 전반적으로 건강한 탐색으로 보입니다.

- 정리하면, 지표 하나하나가 서로 다른 것을 알려 주지만 맥락 안에서 함께 보는 것이 쓸모 있습니다.

&nbsp;
## 7.4 정책 비율 자르기로 시퀀스 단위 GRPO 안정시키기

- 지금까지는 주로 GRPO 학습 결과를 분석했습니다.
- 이제부터는 GRPO 알고리즘에 기능을 더해 갑니다.
- 지금까지 쓴 GRPO 알고리즘은 단순화한 버전입니다. 이 절에서는 GRPO 손실에 잘린 정책 비율(clipped policy ratio)을 쓰는 데 집중합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F10_raschka.webp" width=400>

&nbsp;
### 7.4.1 잘린 정책 비율 계산하기

- 잘린 정책 비율은 지금의 정책(학습 중인 LLM)이 같은 정책의 예전 버전과 얼마나 다른지를 잽니다.
- 구체적으로는 업데이트 스텝 전에 계산한 시퀀스 로그 확률과 업데이트 뒤에 계산한 시퀀스 로그 확률을 비교합니다.
- 아래 GRPO 흐름에서는 4단계의 로그 확률(예전 파라미터로 계산)과 6단계 뒤 업데이트된 모델이 만든 로그 확률을 비교하는 것에 해당합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F11_raschka.webp?1" width=400>

- 복습하자면, 앞 장에서는 정책 경사 손실(policy gradient loss)을 아래처럼 계산했습니다.

In [ ]:
# Compute rollouts
# ...

# Compute rewards
# 롤아웃 4개의 보상 예시: 맞음 2개, 틀림 2개
rewards = torch.tensor([1., 1., 0., 0.])

# Compute sequence logprobs
logprobs = torch.tensor([-7.9243, -20.1546, -16.6130, -23.3677])

advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

pg_loss = -(advantages.detach() * logprobs).mean()
print("Policy gradient loss:", pg_loss)

- 잘린 비율은 예전 정책의 로그 확률(`old_logps`)과 지금의 로그 확률(`new_logps`) 사이에서 계산합니다.
- 유도 과정은 이 책의 범위를 벗어납니다. 궁금한 분은 [Proximal Policy Optimization Algorithms](https://arxiv.org/pdf/1707.06347) 논문에서 더 알아볼 수 있습니다.
- 설명을 위해 위 예시의 로그 확률을 `old_logps`로 쓰고, 모델을 한 번 업데이트했다고 치고 업데이트된 모델로 `new_logps`를 계산합니다(예전 정책과 지금 정책에 같은 프롬프트를 넣어야 공정하게 비교할 수 있습니다).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F12_raschka.webp?2" width=600>

In [ ]:
new_logps = logprobs

# The values of the new_logps
# are shown side by side in the comments
old_logps = torch.tensor([
    -10.9243,   # -7.9243
    -20.3546,   # -20.1546
    -14.6130,   # -16.6130
    -23.3677,   # -23.3677
])

# 로그 확률 차이에 exp를 씌우면 새 확률 / 예전 확률 비율이 됩니다
log_ratio = new_logps - old_logps
ratio = torch.exp(log_ratio)
clip_eps = 10.0
# 비율을 1 ± clip_eps 범위 안으로 자릅니다
clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)

print("Ratio:        ", ratio)
print("Clipped ratio:", clipped_ratio)

- `ratio`는 예전 로그 확률과 새 로그 확률이 얼마나 다른지 알려 줍니다. 둘이 같으면 1.0이 나옵니다.
- `clipped_ratio`는 예전 정책에서 벗어날 수 있는 최대 폭을 정해 둡니다. 업데이트를 비교적 조심스럽게 유지해 학습을 안정시키려는 것입니다.
- 흔히 1 +/- `clip_eps`로 자릅니다. DeepSeek-R1 학습에서는 clip_eps를 10으로 잡았는데, 이는 아주 약하게 자른다는 뜻입니다(다른 강화학습 파이프라인에서는 `clip_eps`를 0.1처럼 훨씬 작게 잡는 일도 드물지 않습니다).
  

- 다음으로 `ratio`와 `clipped_ratio`를 손실 계산에 넣습니다.
- 전에는 어드밴티지와 로그 확률을 바로 곱했습니다.
- 이제는 어드밴티지에 정책 비율을 곱하고, 잘린 목적 함수(clipped objective)를 써서 롤아웃 하나가 업데이트에 미치는 영향을 제한합니다.

In [ ]:
# Treat advantages as fixed learning signals (no backprop through rewards)
adv = advantages.detach()

unclipped = ratio * adv
clipped = clipped_ratio * adv

# PPO-style clipping
# 자른 값과 안 자른 값 중 작은 쪽을 써서 업데이트를 보수적으로 만듭니다
obj = torch.minimum(unclipped, clipped)

    
clipped_pg_loss = -torch.mean(obj)
policy_ratio = torch.mean(ratio)

print("Clipped policy gradient loss:", clipped_pg_loss)
print("Policy ratio:", policy_ratio)

- 이 예시에서 정책 비율이 크다는 것은, 업데이트된 정책(모델)이 뽑힌 토큰에 예전 정책보다 훨씬 높은 확률을 준다는 뜻입니다.
- 잘린 정책 경사 손실(-2.3998)이 앞에서 계산한 일반 정책 경사 손실(-2.5764)보다 조금 작아졌습니다. 모델 가중치 업데이트가 전보다 조금 작아진다는 뜻입니다.
- 이렇게 자르면 학습을 흔들 수 있는 지나치게 공격적인 업데이트를 막을 수 있습니다.

- 참고로 GRPO에서 어떤 방법은 토큰 단위 확률을, 어떤 방법은 시퀀스 단위 확률을 씁니다. 흔한 변형을 정리하면 아래와 같습니다.

| 논문 | 방식 | 표기 |
|---|---|---|
| [DeepSeekMath GRPO](https://arxiv.org/abs/2402.03300), 2024 | 토큰 단위 | `r_{i,t}(theta) = pi_theta(o_{i,t} \| q, o_{i,<t}) / pi_theta_old(o_{i,t} \| q, o_{i,<t})` |
| [DeepSeek-R1 GRPO](https://arxiv.org/abs/2501.12948), 2025 | 시퀀스 단위 | `r_i(theta) = pi_theta(o_i \| q) / pi_theta_old(o_i \| q)` |
| [DAPO](https://arxiv.org/abs/2503.14476), 2025 | 토큰 단위 | `r_{i,t}(theta) = pi_theta(o_{i,t} \| q, o_{i,<t}) / pi_theta_old(o_{i,t} \| q, o_{i,<t})` |
| [GSPO](https://arxiv.org/abs/2507.18071), 2025 | 시퀀스 단위 | `s_i(theta) = (pi_theta(y_i \| x) / pi_theta_old(y_i \| x))^(1 / \|y_i\|)` |

&nbsp;
### 7.4.2 잘린 정책 비율로 학습하기

- 잘린 정책 비율 계산을 넣은 새 GRPO 학습 스크립트는 부록 자료에서 내려받을 수 있습니다.

In [ ]:
# 정책 비율 자르기를 넣은 스크립트를 내려받습니다
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_4_plus_clip_ratio.py"
)

- 앞에서처럼 아래와 같이 돌릴 수 있습니다.

```bash
uv run 7_4_plus_clip_ratio.py \
    --steps 500  \
    --max_new_tokens 1024
```

- DeepSeek-R1은 롤아웃을 아주 많이(8,192개) 만들어 미니배치 16개로 나눈 뒤, 그 미니배치들을 안쪽 에포크(inner epoch) 한 번으로 훑습니다.
- 구체적인 방법은 다음과 같습니다.
  1. 지금 모델을 복사해 기준 정책(reference policy)으로 둡니다.
  2. 기준 정책으로 큰 롤아웃 묶음을 뽑습니다.
  3. 그렇게 고정한 롤아웃을 미니배치로 나눕니다.
  4. 미니배치마다:
      - 기준 모델로 `old_logps`를 계산합니다.
      - 지금 모델(미니배치마다 업데이트되어 바뀝니다)로 `new_logps`를 계산합니다.
      - 지금 모델을 업데이트합니다.
  5. 기준 모델을 업데이트합니다.

- 자원이 넉넉하지 않아서 그렇게 많은 롤아웃과 미니배치를 만들 수는 없습니다. 그래서 우회 방법을 씁니다.
1. 지금 모델을 복사해 기준 정책으로 둡니다.
2. 기준 정책으로 롤아웃 8개를 뽑습니다.
3. 그다음 계산합니다.
  - 기준 모델로 `old_logps`
  - 지금 모델로 `new_logps`
  - 지금 모델 업데이트
4. 미니배치를 쓰는 대신, 다른 롤아웃으로 2단계와 3단계를 되풀이합니다.
5. 기준 모델을 업데이트합니다.

- 짧게 말해 DeepSeek-R1은 거대한 롤아웃 묶음을 한 번 만들고 미니배치로 나눠 훑습니다.
- 우리 코드는 작은 롤아웃 묶음을 여러 번 만듭니다.

- 이 실행의 로그 파일은 아래처럼 내려받아 그래프로 볼 수 있습니다.

In [ ]:
download_from_github(
    "ch07/02_logs/7_4_plus_clip_ratio_metrics.csv",
)

In [ ]:
# 기본 지표 4개를 그립니다
plot_grpo_metrics(
    "7_4_plus_clip_ratio_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    save_as="13_1.pdf"
)

In [ ]:
# 정책 비율과 어드밴티지, 엔트로피를 그립니다
plot_grpo_metrics(
    "7_4_plus_clip_ratio_metrics.csv",
    columns=["policy_ratio", "adv_avg", "adv_std", "entropy_avg"],
    save_as="13_2.pdf"
)

- 위 그래프를 보면 400 스텝 무렵의 하락이 더는 없습니다. 학습이 전보다 안정적이 되었습니다.

&nbsp;
## 7.5 KL 항으로 모델이 바뀌는 정도 조절하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F14_raschka.webp?2" width=400>

- 앞 장은 이미 아주 길어서, GRPO 알고리즘의 KL 손실 항은 건너뛰었습니다.
- 강화학습 알고리즘을 아는 분이라면, 우리가 사실상 그룹 정규화 어드밴티지를 쓰는 [REINFORCE](https://en.wikipedia.org/wiki/Policy_gradient_method#REINFORCE)를 구현했다는 것을 알아챘을 것입니다.
- GRPO 알고리즘을 마저 완성하려고, 이제 KL 손실 항을 더합니다.
- KL은 쿨백-라이블러 발산(Kullback-Leibler divergence)의 줄임말로, KL 발산(KL divergence)이라고도 부릅니다. 지금의 정책(학습 중인 LLM)이 기준 정책(학습을 시작할 때의 원래 LLM)에서 얼마나 벗어났는지를 잽니다.
- 지나치게 큰 정책 업데이트를 억제하는 규제(regularizer) 역할을 합니다. 덕분에 모델이 원래 행동에서 크게 벗어나지 않고, 급격하고 불안정한 업데이트를 막을 수 있습니다.
- 앞에서 구현한 잘린 정책 비율과도 관련이 있습니다. 전에는 업데이트 한 걸음의 크기에만 집중했다면, 이제는 오랜 기간에 걸쳐 벗어나는 정도(drift)를 직접 줄입니다.

&nbsp;
### 7.5.1 KL 손실 항 구현하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F15_raschka.webp?2" width=400>

- 롤아웃마다, 똑같이 생성된 토큰으로 지금의 정책(모델)과 고정된 기준 모델 양쪽에서 로그 확률을 계산합니다(그림 오른쪽).
- 그다음 서로 짝이 되는 로그 확률의 차이를 모두 더해 KL 발산을 계산합니다. 이 값이 지금의 정책이 기준에서 얼마나 옮겨 갔는지를 수치로 보여 줍니다.
- 이 KL 항을 정책 경사 손실에 더합니다. 그러면 보상은 늘리지만 기준 모델과의 차이(발산)도 크게 늘리는 가중치 업데이트는 학습 중에 벌점을 받습니다.
- 여기서 기준 모델은 학습을 시작하기 전의 원래 모델입니다(DeepSeek-R1에서는 400 스텝마다 이 기준 모델을 최신 모델로 바꿉니다).

- 코드를 되풀이하지 않으려고, 아래 코드는 고친 `compute_grpo_loss_with_kl` 전체를 보여 주지 않습니다. `compute_grpo_loss` 함수에서 바꿔야 하는 부분에만 집중합니다.
- 새로 들어간 부분은 `"# New"`로 표시했거나 `if kl_coeff` 아래에 있는 부분입니다.

In [ ]:
import copy

# kl_coeff = 0.0 deactivates the KL term and the code is similar to before
kl_coeff = 0.0  

# Make copy of the original model and disable gradients so it isn't updated
# KL 항을 쓸 때만 원래 모델을 복사해 기준 모델로 얼려 둡니다
if kl_coeff:
    ref_model = copy.deepcopy(model).to(device)  # noqa: F821
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False
else:
    ref_model = None


def compute_grpo_loss_with_kl(
    model,
    ref_model,  # New
    # ...
    kl_coeff=0.02,  # New
):
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []  # noqa: F841
    # ...

    for _ in range(num_rollouts):  # noqa: F821
        token_ids, prompt_len, text = sample_response(  # noqa: F821
        # ...
        )

        # Compute logprobs with reference model
        if kl_coeff:
            # 기준 모델은 학습하지 않으니 기울기를 계산하지 않습니다
            with torch.no_grad():
                ref_logp = sequence_logprob(  # noqa: F821
                    ref_model, token_ids, prompt_len
                )
        else:
            ref_logp = None

        reward = reward_rlvr(text, example["answer"])  # noqa: F821

        roll_rewards.append(reward)
        roll_logps.append(logp)  # noqa: F821
        if kl_coeff:
            roll_ref_logps.append(ref_logp)
        
        # ...

    rewards = torch.tensor(roll_rewards, device=device)  # noqa: F841
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    logps = torch.stack(roll_logps)
    if kl_coeff:
        ref_logps = torch.stack(roll_ref_logps).detach()

    # 지금까지와 같은 정책 경사 손실입니다
    pg_loss = -(advantages.detach() * logps).mean()

    # Compute KL loss term
    if kl_coeff:
        # 지금 모델과 기준 모델의 로그 확률 차이에 계수를 곱합니다
        kl_loss = kl_coeff * torch.mean(logps - ref_logps)
    else:
        kl_loss = torch.tensor(0.0, device=logps.device)

    # Add KL loss term to policy gradient loss
    loss = pg_loss + kl_loss  # noqa: F841

- KL 손실 항을 더하면 자원이 더 듭니다. 원래 모델의 복사본을 하나 더 들고 있어야 하기 때문입니다.
- `kl_coeff`가 클수록 발산에 주는 벌점이 커집니다.

&nbsp;
### 7.5.2 KL 손실 항으로 학습하기

- 전체 스크립트는 아래처럼 부록 자료에서 내려받을 수 있습니다.

In [ ]:
# KL 항을 넣은 스크립트를 내려받습니다
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_5_plus_kl.py"
)

- 앞에서처럼 아래와 같이 돌릴 수 있습니다.

```bash
uv run 7_5_plus_kl.py \
    --steps 500  \
    --max_new_tokens 1024
```

- 앞에서처럼 실험을 돌리는 데 자원과 비용이 많이 들 수 있으니, 분석용 로그 파일을 아래처럼 내려받을 수 있습니다.

In [ ]:
# KL 실험의 학습 로그를 내려받습니다
download_from_github(
    "ch07/02_logs/"
    "7_5_plus_kl_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_5_plus_kl_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    # save_as="16.pdf"
)

In [ ]:
plot_grpo_metrics(
    "7_5_plus_kl_metrics.csv",
    columns=["policy_ratio", "adv_avg", "adv_std", "entropy_avg"],
)

- 학습이 50 스텝을 조금 넘기자마자 무너지고, 평가 정확도가 0으로 떨어지는 것을 볼 수 있습니다.
- 모델이 만든 답을 직접 확인해 보면, 모델이 `"framework.raises Self profess(import"` 같은 말도 안 되는 글과 무작위 토큰을 내놓기 시작합니다.
- 여기에는 여러 이유가 있을 수 있습니다.
1. KL 항을 로그 확률의 합으로 계산합니다.
   - `kl_loss = kl_coeff * mean(new_logps - ref_logps)`에서 `new_logps`와 `ref_logps`는 답 토큰 전체에 대한 합입니다.
   - 그래서 시퀀스가 길수록 KL 항의 크기가 커집니다. 이것이 아주 긴 출력을 부추기고(응답 길이가 늘어나는 것으로 보입니다) 학습을 흔들 수 있습니다.
2. 200 스텝 무렵 보상이 무너지면, KL만이 유일한 기울기가 됩니다.
   - 보상이 모두 0이면 어드밴티지도 0이 되어 정책 경사 손실에는 기울기가 없습니다.
   - 하지만 KL 항은 여전히 남아 있어서 업데이트를 지배합니다. 그러면 모델을 더 높은 엔트로피, 거의 고른(uniform) 분포 쪽으로 밀 수 있습니다. 생성된 답이 무작위처럼 보이는 것(그리고 평가 정확도 0.0%)도 이것으로 설명됩니다.

- KL 항은 GRPO의 표준 구성 요소라서 여기서 소개했습니다. 하지만 여러 연구자는 KL 항이 없을 때 모델이 더 잘 학습될 수 있다는 것을 발견했습니다. 예를 들면 다음과 같습니다.
  - [Dr. GRPO](https://arxiv.org/abs/2503.20783)
  - [Olmo 3](https://arxiv.org/abs/2512.13961)
  - [DeepSeek V3.2](https://arxiv.org/abs/2512.02556)

아니면 다른 방법으로 학습을 안정시킬 수도 있습니다.

1. 토큰 단위 로그 확률을 써서 로그 확률과 KL을 길이로 정규화합니다.
   - 토큰 단위 로그 확률을 쓰려면, 정책 비율과 KL 항을 계산하기 전에 생성된 길이로 나눕니다.
   - 이렇게 하면 시퀀스 길이를 부풀릴 동기가 사라집니다.

2. `kl_coeff`를 낮춥니다(예: 0.001. 하지만 해 보니 여기서는 이것도 도움이 되지 않았습니다).

3. 가중치를 다시 매긴 KL 항(중요도 샘플링, importance sampling)을 씁니다.
   - 예전: `kl_loss = kl_coeff * torch.mean(logps - ref_logps)`
   - 새로: `kl_loss = kl_coeff * torch.mean(torch.exp(new_logps - old_logps) * (new_logps - ref_logps))`

4. 작은 형식 보상(format reward)을 더해 어드밴티지가 0으로 무너지지 않게 합니다. 형식 보상은 다음 절에서 다룹니다.

- 하지만 앞에서 말했듯이, 수학 데이터로 학습할 때는 KL 항을 쓰지 말라는 것이 흔한 권장 사항입니다(Dr. GRPO, Olmo 3, DeepSeek V3.2의 권장).
  - 이렇게 하면 코드도 단순해지고 필요한 자원도 줄어듭니다.

&nbsp;
## 7.6 명시적인 형식 보상 더하기

- 지금까지는 정답 보상 하나만 썼습니다. 이 절에서는 DeepSeek-R1처럼 두 번째 보상으로 형식 보상을 더합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F17_raschka.webp?1" width=400>

&nbsp;
### 7.6.1 `<think>` 토큰 쓰기

- 어떤 추론 모델(DeepSeek-R1, Qwen3 등)은 이른바 `<think>` `</think>` 토큰을 씁니다.
- 꼭 필요하지는 않습니다. 하지만 모델이 중간 추론과 최종 답을 나눠 쓰게 하고, 출력 구조를 일정하게 지키게 하는 데 쓸모가 있습니다.
- 앞에서 쓴 기본 모델의 토크나이저를 불러옵시다.

In [ ]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()
# 기본(base) 모델과 그 토크나이저를 불러옵니다
model, tokenizer_base = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 보다시피 `<think>` 토큰을 여러 개의 하위 단어(subword) 토큰으로 쪼갭니다. `<think>` 토큰이 어휘에 없다는 뜻입니다.

In [ ]:
# 기본 토크나이저는 <think>를 여러 조각으로 쪼갭니다
print(tokenizer_base.encode("<think>"))

In [ ]:
# 쪼개진 토큰 ID를 하나씩 글자로 되돌려 봅니다
for i in [13708, 766, 29]:
    print(tokenizer_base.decode([i]))

- 토크나이저에는 비어 있는 자리가 몇 개 있습니다. 그래서 아래처럼 `<think>` 토큰을 어휘에 더할 수 있습니다.

In [ ]:
# 빈 자리에 특수 토큰 4개를 더합니다
tokenizer_base._tok.add_special_tokens(
    ["<tool_response>", "</tool_response>", "<think>", "</think>"]
)

In [ ]:
print(tokenizer_base.encode("<think>"))

In [ ]:
print(tokenizer_base.encode("</think>"))

- 위에서 `"<tool_response>"`와 `"</tool_response>"` 토큰도 함께 더했습니다. 곧 다룰 Qwen3 추론 버전의 토크나이저와 맞추려는 것입니다.

- 모델도 이 토큰들을 받아들일 수 있습니다. 어휘 크기가 151936이라 토큰 ID 0부터 151935까지 받기 때문입니다.

In [ ]:
# 임베딩 행 수 = 모델이 받을 수 있는 어휘 크기
print("Vocabulary size:", model.tok_emb.weight.shape[0])

- 아니면 추론 모델 버전의 토크나이저를 쓰면 됩니다. 이 토크나이저는 처음부터 `<think>` 토큰을 지원합니다.

- 2장에서 한 것처럼 토크나이저만 따로 불러올 수 있습니다.

In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import download_qwen3_small

# 추론(reasoning) 모델의 토크나이저 파일만 내려받습니다
download_qwen3_small(
    kind="reasoning", tokenizer_only=True, out_dir="qwen3"
)

tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [ ]:
# 추론 토크나이저에서는 <think>가 토큰 하나입니다
print(tokenizer.encode("<think>"))

In [ ]:
print(tokenizer.encode("</think>"))

- 그다음 LLM이 `<think>` `</think>` 태그를 쓰는지 확인하는 보상 함수를 만들 수 있습니다.
- 이 함수는 출력에 `<think>`와 `</think>`가 둘 다 바른 순서로 나오면 1.0을, 아니면 0.0을 돌려줍니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F18_raschka.webp?1" width=600>

In [ ]:
def reward_format(
    token_ids,
    prompt_len,
    start_think_id=151667,
    end_think_id=151668,
):
    try:
        # 프롬프트를 뺀 생성 부분만 봅니다
        gen = token_ids[prompt_len:].tolist()
        return float(
            gen.index(start_think_id) < gen.index(end_think_id)
        )
    # 둘 중 하나라도 없으면 index가 오류를 내므로 0점입니다
    except ValueError:
        return 0.0

In [ ]:
prompt = "Calculate ..."
rollout = "Let's ... <think> ... </think> ..."
token_ids = tokenizer.encode(prompt + rollout)

# <think>가 </think>보다 앞에 있으니 1.0이 나와야 합니다
reward_format(
    token_ids=torch.tensor(token_ids),
    prompt_len=len(tokenizer.encode(prompt))
)

- 그러면 형식 보상을 추가 보상 신호로 써서 모델을 학습시킬 수 있습니다.
- 예를 들어 `compute_grpo_loss`를 아래처럼 `compute_grpo_loss_plus_format_reward` 함수로 고칠 수 있습니다.

In [ ]:
def compute_grpo_loss_plus_format_reward(
    # ...
    format_reward_weight=1.0,  # NEW
):
    # ...

        logp = sequence_logprob(model, token_ids, prompt_len)  # noqa: F821, F841
        rlvr_reward = reward_rlvr(text, example["answer"])  # noqa: F821
        format_reward = reward_format(token_ids, prompt_len)         # NEW  # noqa: F821
        # 정답 보상에 가중치를 곱한 형식 보상을 더합니다
        reward = rlvr_reward + format_reward_weight * format_reward  # NEW  # noqa: F841

    # ...

- 여기에 더해, 모델이 `<think>` 토큰을 내놓도록 `render_prompt`도 고치려 합니다.

In [ ]:
def render_prompt_with_think_tokens(prompt):
    # 추론은 <think> 태그 안에, 답은 \boxed{}로 쓰라고 지시합니다
    template = (
        "You are a helpful math assistant.\n"
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n"
        "Then write the final result on a new line in the exact format:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

&nbsp;
### 7.6.2 `<think>` 토큰을 내놓도록 모델 학습하기

- 고친 스크립트는 아래처럼 부록 자료에서 내려받을 수 있습니다.

In [ ]:
# 형식 보상을 넣은 스크립트를 내려받습니다
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py"
)

- 기본 모델을 GRPO로 학습시켜 `<think>` 토큰을 내놓게 하면 결과가 꽤 나쁩니다. 기본 모델은 `<think>` 토큰을 한 번도 본 적이 없기 때문입니다.
- 그 대신 먼저 `<think>` 토큰이 들어간 데이터로 모델을 사전학습하거나 지시 미세조정(instruction fine-tuning)하는 편이 좋습니다(지시 미세조정은 다음 장에서 다룹니다).
- 그래서 이 절에서는 이미 `<think>` 토큰에 익숙한, 기존 "reasoning"(추론) 버전의 모델을 학습시킵니다.

- 앞에서처럼 아래와 같이 돌릴 수 있습니다(이 스크립트는 이미 "base" 모델 대신 "reasoning" 모델 버전을 쓰도록 설정되어 있습니다).

```bash
uv run 7_6_plus_format_reward.py \
    --steps 500  \
    --max_new_tokens 1024
```

- 이번에도 실험을 돌리는 데 자원과 비용이 많이 들 수 있으니, 분석용 로그 파일을 아래처럼 내려받을 수 있습니다.

In [ ]:
# 형식 보상 실험의 학습 로그를 내려받습니다
download_from_github(
    "ch07/02_logs/7_6_plus_format_reward_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    #save_as="18.pdf"
)

> **`loss`와 `kl_loss`에 대한 참고**
>
> - 이 실험은 `7_6_plus_format_reward.py`를 기본 설정 `--kl_coeff 0.0`, `--inner_epochs 1`로 돌렸습니다.
> - `--kl_coeff`가 `0.0`이라 KL 벌점이 꺼져 있습니다. 그래서 기록된 `kl_loss`는 학습 내내 `0`입니다.
>
> - 7.5절과 같은 설정으로 KL 규제를 켜고 싶다면 아래처럼 돌리세요.
>
> ```bash
> uv run 7_6_plus_format_reward.py \
>     --steps 500 \
>     --max_new_tokens 1024 \
>     --kl_coeff 0.001 \
>     --inner_epochs 2
> ```
>
> - 7.6절 기본 설정에서는 기록된 `loss`도 `0`입니다.
> - 안쪽 업데이트를 한 번만 하고(`--inner_epochs 1`), 같은 스텝의 자기 복사본과 비교하기 때문입니다.
> - 그 결과 정책 비율은 `1`에서 시작합니다. 어드밴티지는 평균이 `0`이 되도록 정규화되어 있으니, 스칼라 정책 손실 값은 `0`이 됩니다.
>
> - 그러니 이 경우 `loss = 0`은 기본 설정 때문에 생긴 계산상의 결과일 뿐입니다.
> -  즉 안쪽 에포크를 한 번만 쓰고 KL 항이 없어서, 기록된 스칼라 손실이 아무 정보도 주지 못한다는 것을 주로 보여 줍니다.
> - 여기서 지켜보기에 더 쓸모 있는 값은 `reward_avg`, `format_reward_avg`, `adv_std`, `entropy_avg` 같은 지표입니다.


In [ ]:
# 형식 보상 평균도 함께 그려 봅니다
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
    #save_as="19.pdf"
)

- 보다시피 모델은 처음에 좋아지다가 첫 100 스텝이 지나면 나빠집니다. 첫 100 스텝 뒤 모델이 내놓는 응답 길이가 짧아진 것과 관련이 있을 수 있습니다.
- 한 가지 설명은, 위에서 보듯 모델이 `<think>` 태그 형식을 지키는 것만으로 보상을 너무 많이 받아서 정답에 충분히 집중하지 않는다는 것입니다.
- 기본 `format_reward_weight`를 1.0에서 0.1로 낮춰 형식 보상의 크기를 줄일 수 있습니다.
- 형식 보상을 조건부로 만드는 것도 개선책이 될 수 있습니다. 답이 맞았을 때만 형식 보상을 주는 것입니다(지금은 답이 맞든 틀리든 따로 줍니다).

```python
if conditional_reward:
    format_reward *= rlvr_reward
reward = rlvr_reward + format_reward_weight * format_reward
```

&nbsp;
## 7.7 GRPO의 또 다른 수정, 요령과 팁

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F21_raschka.webp?3" width=400>

- DeepSeek-R1이 성공한 뒤 몇 달 사이, 많은 연구자가 원래 GRPO 알고리즘의 안정성과 성능을 함께 높이는 개선안을 내놓았습니다.
- 이 장이 이미 길어서, 이 절에서는 각 변경·개선 사항을 짧게 나열만 합니다.
- 더 자세한 내용과 코드는 [부록 자료](../03_rlvr_grpo_scripts_advanced/)에서 찾을 수 있습니다.

1. 기울기 신호가 0인 경우 걸러 내기(Zero gradient signal filtering) ([DAPO by Yu et al., 2025](https://arxiv.org/abs/2503.14476))
2. 능동 샘플링(Active sampling) (DAPO)
3. 토큰 단위 손실(Token-level loss) (DAPO)
4. KL 손실 없애기(No KL loss) (DAPO 및 [Dr. GRPO by Liu et al., 2025](https://arxiv.org/abs/2503.20783))
5. 위쪽 더 넓게 자르기(Clip higher) (DAPO)
6. 잘린 중요도 샘플링(Truncated importance sampling) ([Yao et al., 2025](https://fengyao.notion.site/off-policy-rl))
7. 표준편차 정규화 없애기(No standard deviation normalization) (Dr. GRPO)
8. 분야별 KL 세기로 KL 조정, 수학은 0(KL tuning with domain-specific KL strengths; zero for math) ([DeepSeek V3.2](https://arxiv.org/abs/2512.02556)
9. 가중치를 다시 매긴 KL(Reweighted KL) (DeepSeek V3.2)
10. 오프폴리시 시퀀스 가리기(Off-policy sequence masking) (DeepSeek V3.2)
11. top-p / top-k 샘플링 마스크 유지(Keep sampling mask for top-p / top-k) (DeepSeek V3.2)
12. 원래 GRPO 어드밴티지 정규화 유지(Keep original GRPO advantage normalization) (DeepSeek V3.2)
13. 보상을 합치기 전에 보상별로 그룹 정규화(Per-reward group-wise normalization before aggregation) ([GDPO by Liu et al., 2026](https://arxiv.org/abs/2601.05242))
14. 시퀀스 단위 중요도 샘플링과 자르기(Sequence-level importance sampling and clipping) ([GSPO by Zheng et al., 2025](https://arxiv.org/abs/2507.18071))
15. 토큰 업데이트 대신 중요도 샘플링 가중치 자르기(Clip importance-sampling weights rather than token updates) ([CISPO by MiniMax et al., 2025](https://arxiv.org/abs/2506.13585))

&nbsp;
## 7.8 정리

- 이 절에는 코드가 없습니다.